# CyberPhi — Model Inference & Capability Testing

**Purpose:** Load a pre-trained CyberPhi model and test its cybersecurity reasoning capabilities.
This notebook does **not** run training or the data pipeline — it only loads weights and runs inference.

**Supported weight sources:**
- **LoRA adapter** (`adapter_model.safetensors` + `adapter_config.json`) from RunPod or local training
- **Merged weights** (full HF model directory with safetensor shards) after `merge_and_export`
- **Upload** — zip your adapter/model folder and upload directly

**Before running:**
1. `Runtime → Change runtime type → GPU` (T4 works for LoRA adapter and merged Phi-3.5-mini)
2. If loading from Drive: mount in Cell 3 and set `DRIVE_OUTPUT_PATH`
3. Run cells top to bottom

---
## 0 — GPU Check

In [ ]:
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError('No GPU detected — go to Runtime → Change runtime type and select GPU')

gpu_name, vram = [s.strip() for s in result.stdout.strip().split(',')]
print(f'GPU:  {gpu_name}')
print(f'VRAM: {vram}')

HIGH_END_GPUS = ('A100', 'A10', 'L4', 'H100')
IS_HIGH_END   = any(g in gpu_name for g in HIGH_END_GPUS)
print(f'\nProfile: {"high-end (bfloat16)" if IS_HIGH_END else "T4/P100 (float16 + 4-bit QLoRA)"}')

---
## 1 — Install Dependencies

Colab ships with `torch`, `transformers`, `peft`, `accelerate`, and `bitsandbytes`.
We only add `rouge-score` if it's missing.

In [ ]:
!pip install rouge-score -q

import torch, transformers, peft
print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')
print(f'CUDA:         {torch.cuda.is_available()}')

---
## 2 — Mount Google Drive (optional)

Set `LOAD_FROM_DRIVE = True` if your model files are in Google Drive (e.g. synced from RunPod).
The default `DRIVE_OUTPUT_PATH` matches where the quickstart notebook and RunPod training write outputs.

In [ ]:
LOAD_FROM_DRIVE   = True   # @param {type:"boolean"}
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/output"  # @param {type:"string"}

if LOAD_FROM_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print(f'Drive mounted. Model search root: {DRIVE_OUTPUT_PATH}')
else:
    print('Drive not mounted — use LOAD_MODE="upload" or point paths at /content/...')

---
## 3 — Model Source Configuration

Choose how to load the model:
- **`lora_adapter`** — LoRA adapter on top of the base model (4-bit, runs on T4)
- **`merged_weights`** — Full merged model in HF format (bfloat16/float16, also runs on T4 for Phi-3.5-mini)
- **`upload`** — Upload a zip of your adapter or merged model folder

In [ ]:
LOAD_MODE  = "lora_adapter"  # @param ["lora_adapter", "merged_weights", "upload"]
BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"  # @param {type:"string"}

# Set these to the exact folder containing adapter_config.json / config.json.
# Common locations after training:
#   colab_quickstart output  →  {DRIVE_OUTPUT_PATH}/colab_test
#   RunPod / Axolotl output  →  {DRIVE_OUTPUT_PATH}/cyberphi-lora
#   Merged weights           →  {DRIVE_OUTPUT_PATH}/cyberphi-lora/merged
ADAPTER_PATH = f"{DRIVE_OUTPUT_PATH}/colab_test"             # @param {type:"string"}
MODEL_PATH   = f"{DRIVE_OUTPUT_PATH}/cyberphi-lora/merged"   # @param {type:"string"}
UPLOAD_PATH  = "/content/model"

print(f'Mode:       {LOAD_MODE}')
print(f'Base model: {BASE_MODEL}')
if LOAD_MODE == 'lora_adapter':
    print(f'Adapter:    {ADAPTER_PATH}')
elif LOAD_MODE == 'merged_weights':
    print(f'Model dir:  {MODEL_PATH}')
else:
    print(f'Upload to:  {UPLOAD_PATH}')

---
## 4 — Load Model & Tokenizer

In [ ]:
import os, zipfile, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

SYSTEM = 'You are a senior cybersecurity expert and penetration tester.'


def _find_adapter_root(path):
    """Return the directory containing adapter_config.json, searching one level deep."""
    if os.path.isfile(os.path.join(path, 'adapter_config.json')):
        return os.path.abspath(path)
    if os.path.isdir(path):
        for entry in sorted(os.scandir(path), key=lambda e: e.name):
            if entry.is_dir() and os.path.isfile(os.path.join(entry.path, 'adapter_config.json')):
                print(f'  Found adapter_config.json in: {entry.path}')
                return os.path.abspath(entry.path)
    return None


# ── Handle upload mode ────────────────────────────────────────────────────────
if LOAD_MODE == 'upload':
    from google.colab import files as colab_files
    print('Upload your adapter zip or merged model zip:')
    uploaded = colab_files.upload()
    os.makedirs(UPLOAD_PATH, exist_ok=True)
    for fname in uploaded:
        dest = os.path.join('/content', fname)
        if fname.endswith('.zip'):
            with zipfile.ZipFile(dest) as z:
                z.extractall(UPLOAD_PATH)
            print(f'Extracted {fname} → {UPLOAD_PATH}')
        else:
            os.rename(dest, os.path.join(UPLOAD_PATH, fname))
    if os.path.exists(os.path.join(UPLOAD_PATH, 'adapter_config.json')):
        LOAD_MODE    = 'lora_adapter'
        ADAPTER_PATH = UPLOAD_PATH
        print('Detected LoRA adapter — switching to lora_adapter mode')
    else:
        LOAD_MODE  = 'merged_weights'
        MODEL_PATH = UPLOAD_PATH
        print('Detected merged model — switching to merged_weights mode')

# ── Validate paths before loading ─────────────────────────────────────────────
if LOAD_MODE == 'lora_adapter':
    resolved = _find_adapter_root(ADAPTER_PATH)
    if resolved is None:
        raise FileNotFoundError(
            f"adapter_config.json not found at '{ADAPTER_PATH}' or its subdirectories.\n\n"
            f"Fix: update ADAPTER_PATH in Cell 3 to the folder that contains adapter_config.json.\n"
            f"Common locations:\n"
            f"  colab_quickstart run  →  {{DRIVE_OUTPUT_PATH}}/colab_test\n"
            f"  RunPod / Axolotl run  →  {{DRIVE_OUTPUT_PATH}}/cyberphi-lora\n\n"
            f"You can check what's in your Drive output folder with:\n"
            f"  import os; print(os.listdir('{ADAPTER_PATH}'))"
        )
    ADAPTER_PATH = resolved
    print(f'Adapter path resolved: {ADAPTER_PATH}')

elif LOAD_MODE == 'merged_weights':
    abs_model = os.path.abspath(MODEL_PATH)
    if not os.path.isfile(os.path.join(abs_model, 'config.json')):
        raise FileNotFoundError(
            f"config.json not found at '{MODEL_PATH}'.\n"
            f"Fix: update MODEL_PATH in Cell 3 to the merged model directory."
        )
    MODEL_PATH = abs_model
    print(f'Model path resolved: {MODEL_PATH}')

# ── Tokenizer ─────────────────────────────────────────────────────────────────
print(f'Loading tokenizer from {BASE_MODEL}…')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# ── Model ─────────────────────────────────────────────────────────────────────
if LOAD_MODE == 'lora_adapter':
    print(f'Loading base model in 4-bit QLoRA from {BASE_MODEL}…')
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if IS_HIGH_END else torch.float16,
    )
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_cfg, device_map='auto', trust_remote_code=True,
    )
    print(f'Attaching LoRA adapter from {ADAPTER_PATH}…')
    model = PeftModel.from_pretrained(base, ADAPTER_PATH)

elif LOAD_MODE == 'merged_weights':
    dtype = torch.bfloat16 if IS_HIGH_END else torch.float16
    print(f'Loading merged model from {MODEL_PATH} ({dtype})…')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, device_map='auto', torch_dtype=dtype, trust_remote_code=True,
    )

model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f'\nParameters:  {total_params / 1e9:.2f}B')
print(f'Device map:  {model.hf_device_map if hasattr(model, "hf_device_map") else "auto"}')
print('Model ready ✓')

---
## 5 — Inference Helpers

Uses the same ChatML format as training (`axolotl_config.yaml` and `colab_quickstart.ipynb`).

In [ ]:
import re

EOS_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')


def generate(instruction, input_text='', temperature=0.7, max_new_tokens=1024):
    """Generate a response using the ChatML format used during training."""
    user_content = instruction + (f'\n{input_text}' if input_text else '')
    prompt = (
        f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
        f'<|im_start|>user\n{user_content}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=max(temperature, 1e-6),
            do_sample=temperature > 0,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=EOS_ID,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


def _gen_with_system(system, user, context='', temperature=0.7, max_new_tokens=512):
    """Generate with a custom system prompt (used for the multi-step thinking loop)."""
    user_full = user + (f'\n\n[Context]\n{context}' if context else '')
    prompt = (
        f'<|im_start|>system\n{system}<|im_end|>\n'
        f'<|im_start|>user\n{user_full}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=max(temperature, 1e-6),
            do_sample=temperature > 0,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=EOS_ID,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


def extract_think(text):
    """Split output into (think_block, final_answer)."""
    if '</think>' in text:
        parts  = text.split('</think>', 1)
        think  = parts[0].replace('<think>', '').strip()
        answer = parts[1].strip()
    else:
        think  = ''
        answer = text.strip()
    return think, answer


THINK_RE = re.compile(r'<think>\s*.+?\s*</think>', re.DOTALL | re.IGNORECASE)

print('generate(), _gen_with_system(), extract_think() ready ✓')

---
## 6 — Interactive Single Prompt Test

Edit `INSTRUCTION` and optionally `INPUT_CODE` in the form fields below, then run the cell.

In [ ]:
INSTRUCTION     = "Explain how SQL injection works and show a vulnerable PHP code example."  # @param {type:"string"}
INPUT_CODE      = ""  # @param {type:"string"}
TEMPERATURE     = 0.7  # @param {type:"number"}
MAX_NEW_TOKENS  = 1024  # @param {type:"integer"}

response = generate(INSTRUCTION, INPUT_CODE, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
think, answer = extract_think(response)

print('=' * 64)
print(f'INSTRUCTION: {INSTRUCTION}')
if INPUT_CODE:
    print(f'INPUT CODE:  {INPUT_CODE[:80]}')
print('=' * 64)

if think:
    print('\n--- <think> block ---')
    print(think[:1200] + ('…' if len(think) > 1200 else ''))
    print()

print('--- Final Answer ---')
print(answer)
print(f'\n[Total output: {len(response)} chars | Think block: {"yes" if think else "no"}]')

---
## 7 — Vulnerability Type Coverage Test

Runs one prompt per vulnerability class (all 10 types in the training schema).
Reports whether each response contains a `<think>` reasoning block.

In [ ]:
COVERAGE_MAX_TOKENS = 512  # @param {type:"integer"}

VULN_PROMPTS = {
    'sqli':            'Explain SQL injection and write a test payload for a login form.',
    'xss':             'Describe a reflected XSS attack and give a JavaScript payload example.',
    'rce':             'What is remote code execution? Show a Python deserialization RCE proof-of-concept.',
    'ssrf':            'Explain SSRF and how an attacker can pivot to internal services using it.',
    'xxe':             'What is XXE injection? Write a malicious XML payload that reads /etc/passwd.',
    'deserialization': 'Explain insecure deserialization and show a Java gadget chain exploit example.',
    'buffer_overflow': 'Describe a stack buffer overflow vulnerability and the steps to exploit it.',
    'race_condition':  'What is a TOCTOU race condition? Give a concrete exploit scenario.',
    'idor':            'Explain IDOR and show how to test for it systematically in a REST API.',
    'path_traversal':  'What is path traversal? Write a payload to read /etc/passwd via directory traversal.',
}

results_coverage = []
print(f'{"Vuln Type":<22} {"Think?":<10} {"Output len"}')
print('-' * 46)

for vuln_type, prompt in VULN_PROMPTS.items():
    resp      = generate(prompt, max_new_tokens=COVERAGE_MAX_TOKENS)
    has_think = bool(THINK_RE.search(resp))
    results_coverage.append({
        'vuln_type': vuln_type,
        'prompt':    prompt,
        'output':    resp,
        'has_think': has_think,
    })
    print(f'{vuln_type:<22} {"yes" if has_think else "no":<10} {len(resp)}')

think_rate = sum(r['has_think'] for r in results_coverage) / len(results_coverage)
print(f'\nThink block rate: {think_rate:.1%}  ({sum(r["has_think"] for r in results_coverage)}/{len(results_coverage)})')

---
## 8 — Eval on Held-Out Dataset (optional)

If `data/final/validated.jsonl` is accessible (Drive-mounted or uploaded), this cell computes
ROUGE-L, `<think>` block rate, and CoT step distribution against the gold outputs.

Update `EVAL_JSONL` to point at your file.

In [ ]:
EVAL_JSONL = "/content/drive/MyDrive/output/data/final/validated.jsonl"  # @param {type:"string"}
EVAL_LIMIT = 20  # @param {type:"integer"}

import json
from pathlib import Path

_eval_path = Path(EVAL_JSONL)
if not _eval_path.exists():
    print(f'Eval file not found: {EVAL_JSONL}')
    print('Skipping — mount Drive or update EVAL_JSONL to point at your validated.jsonl')
else:
    samples = [json.loads(l) for l in _eval_path.read_text().splitlines() if l.strip()][:EVAL_LIMIT]
    print(f'Loaded {len(samples)} eval samples from {EVAL_JSONL}')

    # ── Metric helpers (inlined to avoid repo import dependency) ───────────────
    def _lcs_len(x, y):
        m, n = len(x), len(y)
        prev = [0] * (n + 1)
        for i in range(1, m + 1):
            curr = [0] * (n + 1)
            for j in range(1, n + 1):
                curr[j] = prev[j-1]+1 if x[i-1]==y[j-1] else max(prev[j], curr[j-1])
            prev = curr
        return prev[n]

    def rouge_l(hyp, ref):
        try:
            from rouge_score import rouge_scorer
            sc = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
            return sc.score(ref, hyp)['rougeL'].fmeasure
        except Exception:
            h, r = hyp.lower().split(), ref.lower().split()
            if not h or not r: return 0.0
            lcs = _lcs_len(h, r)
            p, rec = lcs / len(h), lcs / len(r)
            return 2*p*rec/(p+rec) if (p+rec) else 0.0

    step_re = re.compile(r'(?m)^(?:\d+[.)\s]|[-*]\s)')

    def cot_dist(outputs):
        buckets = {'0': 0, '1-2': 0, '3-5': 0, '6+': 0}
        for o in outputs:
            n = len(step_re.findall(o))
            k = '0' if n == 0 else '1-2' if n <= 2 else '3-5' if n <= 5 else '6+'
            buckets[k] += 1
        return buckets

    # ── Run eval ───────────────────────────────────────────────────────────────
    outputs_eval, refs_eval, classes_eval = [], [], []
    print(f'\n{"#":<4} {"Source":<12} {"Vuln":<16} {"ROUGE-L":<10} {"Think?"}')
    print('-' * 54)

    for i, s in enumerate(samples):
        resp = generate(s['instruction'], s.get('input', ''), temperature=0, max_new_tokens=512)
        outputs_eval.append(resp)
        refs_eval.append(s['output'])
        classes_eval.append(s.get('vuln_type', ''))
        rl        = rouge_l(resp, s['output'])
        has_think = bool(THINK_RE.search(resp))
        print(f'{i+1:<4} {s.get("source","?"):<12} {s.get("vuln_type","?"):<16} {rl:<10.3f} {"yes" if has_think else "no"}')

    rouge_scores  = [rouge_l(o, r) for o, r in zip(outputs_eval, refs_eval)]
    think_hits    = sum(1 for o in outputs_eval if THINK_RE.search(o))

    print(f'\n=== Eval Summary ({len(samples)} samples) ===')
    print(f'Mean ROUGE-L:       {sum(rouge_scores)/len(rouge_scores):.3f}')
    print(f'<think> block rate: {think_hits/len(outputs_eval):.1%}')
    print(f'CoT step dist:      {cot_dist(outputs_eval)}')

    eval_results = [
        {'instruction': s['instruction'], 'output': o, 'expected': r, 'vuln_type': c,
         'rouge_l': rouge_l(o, r), 'has_think': bool(THINK_RE.search(o))}
        for s, o, r, c in zip(samples, outputs_eval, refs_eval, classes_eval)
    ]

---
## 9 — 3-Step Thinking Loop

Implements the same think → reflect → answer pipeline as `inference/thinking_loop.py`,
but running directly against the loaded HF model (no Ollama required).

In [ ]:
THINK_QUERY = "How would you exploit a blind SQL injection vulnerability to extract the full database schema?"  # @param {type:"string"}

SYSTEM_THINK   = 'You are a senior pentester. Think step-by-step through the security problem inside <think></think> tags. Be thorough and methodical.'
SYSTEM_REFLECT = 'You are a critical security reviewer. Identify any gaps, errors, or missing steps in the reasoning provided. Be concise.'
SYSTEM_ANSWER  = 'You are a senior cybersecurity expert. Produce a final, clear, actionable answer using the reasoning and critique below.'

print(f'Query: {THINK_QUERY}\n')
print('=' * 64)

# Step 1 — Think
print('Step 1 — THINKING…')
thinking = _gen_with_system(SYSTEM_THINK, THINK_QUERY, max_new_tokens=700)
print(thinking[:900] + ('…' if len(thinking) > 900 else ''))
print()

# Step 2 — Reflect
print('Step 2 — REFLECTING…')
reflection = _gen_with_system(
    SYSTEM_REFLECT,
    f'Critique the following reasoning about: {THINK_QUERY}\n\n{thinking}',
    max_new_tokens=400,
)
print(reflection[:600] + ('…' if len(reflection) > 600 else ''))
print()

# Step 3 — Answer
print('Step 3 — FINAL ANSWER…')
answer_context = f'Reasoning:\n{thinking}\n\nCritique:\n{reflection}'
final_answer = _gen_with_system(
    SYSTEM_ANSWER, THINK_QUERY, context=answer_context, max_new_tokens=700,
)
print(final_answer)

thinking_loop_result = {
    'query':    THINK_QUERY,
    'thinking': thinking,
    'reflect':  reflection,
    'answer':   final_answer,
}

---
## 10 — Save & Download Results

Collects all test outputs into a single JSON file and offers a browser download.

In [ ]:
import json
from google.colab import files as colab_files

all_results = {
    'model': {
        'load_mode':   LOAD_MODE,
        'base_model':  BASE_MODEL,
        'adapter_path': ADAPTER_PATH if LOAD_MODE == 'lora_adapter' else None,
        'model_path':   MODEL_PATH   if LOAD_MODE == 'merged_weights' else None,
    },
    'vulnerability_coverage': results_coverage,
    'thinking_loop':          thinking_loop_result,
}

# Include eval results if Cell 8 ran successfully
try:
    all_results['eval_set'] = eval_results
except NameError:
    pass

OUT_PATH = '/content/inference_results.json'
with open(OUT_PATH, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'Saved to {OUT_PATH}')
print(f'Sections: {list(all_results.keys())}')
colab_files.download(OUT_PATH)